# 10.2 Xyce 应用角色与 Adapter

## 本节目标

- 理解 wrapper 如何映射 Xyce linear solver 概念
- 追踪 `solve(A,b,x)` 到 Ascend-GMRES backend

## 环境检查

直接检查 Ascend NPU、CANN、acl_rtc 与 CMake 环境；若失败，先加载目标节点的 CANN 工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("说明：本章默认运行 Xyce Adapter benchmark，不宣称完整 upstream Xyce 仿真。")


## README 定义的调用链

`XyceApplicationWrapper` 用 `b=A*x_true` 表示 matrix/RHS assembly；`XyceSparseMatrixAdapter` 保存 CSR；`XyceLinearSolverAdapter` 把 solver kind 映射到 Ascend-GMRES 的 CPU single、OpenMP16 或 `AscendDevice`（Device GMRES）求解器。这样可以研究 solver 替换，而不修改上游 Xyce。

它没有解析电路 netlist，也没有运行 Xyce executable。因此结论应写成“Xyce linear solver wrapper/adapter benchmark”。

## 与 Ascend-GMRES 的关系

adapter 不复制求解算法：`make_solver()` 直接选择依赖工程提供的 solver factory。GMRES 参数保持 restart=30、max iterations=10000、tolerance=1e-6。课程副本 vendored 了 README 对应版本，确保构建不依赖相邻工作区。

## 课后实践

指出 assembly、prepare、solve、correctness 分别位于哪个类/函数，并说明 wrapper 与完整 Xyce 仿真的差别。参考答案见 `answer/10.02_answer.md`。

## 实验记录与练习

沿 Xyce workload → adapter → distributed_gmres_npu → Ascend C kernel 检查真实调用链；历史 HostPrototype CSV 不能作为本次 NPU 实测。

完成后回答：实际后端是什么？reference 与 tolerance 是什么？主要耗时来自计算、通信、传输还是同步？改变一个并发或算法参数后，正确性和性能如何变化？参考答案仅通过本章 `answer/` 链接查阅。
